# Submissão 2 - Modelo B (Novo LSTM c/ features Tf-Idf + Handcrafted)
Notebook para ler o dataset de validação externa e exportar previsões pelo novo modelo PyTorch guardado.

In [ ]:
import re
import numpy as np
import pandas as pd
from collections import Counter
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import pickle
import os

In [ ]:
stop_words = {
    "a", "an", "the", "and", "or", "but", "if", "while",
    "with", "without", "in", "on", "at", "by", "for", "to", "from",
}

def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"\d+", "", text)
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def preprocess_text_clean(text):
    text = str(text).lower()
    text = "".join(char for char in text if char.isalnum() or char.isspace())
    return text

def extract_features(raw_text, clean_text):
    words = clean_text.split()
    num_words = len(words)
    raw_lower = raw_text.lower()

    sentence_split = raw_text.replace("!", ".").replace("?", ".").split(".")
    sentences = [s.strip() for s in sentence_split if s.strip()]
    sentence_lengths = [len(preprocess_text_clean(s).split()) for s in sentences]

    word_counts = Counter(words)
    punctuation = {p: raw_text.count(p) for p in [",", ".", ";", ":"]}
    n_chars = len(raw_text) if raw_text else 1

    features = {}
    features["length_chars"] = len(raw_text)
    features["num_words"] = num_words
    features["num_sentences"] = len(sentences)
    features["avg_word_length"] = np.mean([len(w) for w in words]) if num_words > 0 else 0.0
    features["avg_sentence_length"] = np.mean(sentence_lengths) if sentence_lengths else 0.0
    features["sentence_length_std"] = np.std(sentence_lengths) if len(sentence_lengths) > 1 else 0.0
    features["type_token_ratio"] = len(set(words)) / num_words if num_words > 0 else 0.0
    features["hapax_ratio"] = sum(1 for c in word_counts.values() if c == 1) / num_words if num_words > 0 else 0.0
    features["repetition_ratio"] = max(word_counts.values()) / num_words if num_words > 0 else 0.0

    for p, count in punctuation.items():
        key = f"punct_{p}_ratio".replace(".", "dot")
        features[key] = count / n_chars

    original_tokens = [token for token in preprocess_text_clean(raw_lower).split() if token]
    features["stopword_ratio"] = (sum(1 for token in original_tokens if token in stop_words) / len(original_tokens) if original_tokens else 0.0)
    features["uppercase_ratio"] = sum(1 for ch in raw_text if ch.isupper()) / n_chars
    features["digit_ratio"] = sum(1 for ch in raw_text if ch.isdigit()) / n_chars
    features["whitespace_ratio"] = sum(1 for ch in raw_text if ch.isspace()) / n_chars

    n_contractions = len(re.findall(r"\b\w+'\w+\b", raw_text))
    features["contraction_ratio"] = n_contractions / num_words if num_words > 0 else 0.0

    all_caps_words = [w for w in raw_text.split() if len(w) > 1 and w.isupper()]
    features["allcaps_word_ratio"] = len(all_caps_words) / num_words if num_words > 0 else 0.0
    features["missing_space_after_punct"] = len(re.findall(r"[.!?,;:][A-Za-z]", raw_text)) / n_chars
    features["fragment_ratio"] = (sum(1 for sl in sentence_lengths if sl <= 3) / len(sentence_lengths) if sentence_lengths else 0.0)
    features["runon_ratio"] = (sum(1 for sl in sentence_lengths if sl >= 40) / len(sentence_lengths) if sentence_lengths else 0.0)

    raw_word_tokens = re.findall(r"\b\w+\b", raw_lower)
    filler_words = {"tbh", "lol", "omg", "idk", "imo", "btw", "ngl", "smh", "fyi", "like", "just", "really"}
    features["filler_word_ratio"] = (sum(1 for w in raw_word_tokens if w in filler_words) / len(raw_word_tokens) if raw_word_tokens else 0.0)

    ai_connectors = {"furthermore", "moreover", "additionally", "consequently", "nevertheless", "therefore", "thus"}
    features["ai_connector_ratio"] = (sum(1 for w in raw_word_tokens if w in ai_connectors) / len(raw_word_tokens) if raw_word_tokens else 0.0)

    first_person = {"i", "me", "my", "mine", "myself", "we", "us", "our", "ours"}
    features["first_person_ratio"] = (sum(1 for w in raw_word_tokens if w in first_person) / len(raw_word_tokens) if raw_word_tokens else 0.0)

    features["question_density"] = raw_text.count("?") / len(sentences) if sentences else 0.0

    if len(words) > 1:
        bigrams = list(zip(words[:-1], words[1:]))
        features["bigram_uniqueness"] = len(set(bigrams)) / len(bigrams)
    else:
        features["bigram_uniqueness"] = 0.0

    adverbs = {"very", "quite", "extremely", "remarkably", "clearly"}
    features["adverb_ratio"] = (sum(1 for w in raw_word_tokens if w in adverbs) / len(raw_word_tokens) if raw_word_tokens else 0.0)

    modals = {"would", "could", "should", "might", "may", "must", "can", "will", "shall"}
    features["modal_ratio"] = (sum(1 for w in raw_word_tokens if w in modals) / len(raw_word_tokens) if raw_word_tokens else 0.0)

    discourse_connectors = {"however", "therefore", "furthermore", "moreover", "consequently"}
    features["transition_ratio"] = (sum(1 for w in raw_word_tokens if w in discourse_connectors) / len(raw_word_tokens) if raw_word_tokens else 0.0)

    be_verbs = {"is", "was", "are", "were", "be", "been", "being"}
    passive_constructions = sum(1 for w in raw_word_tokens if w in be_verbs)
    features["passive_voice_ratio"] = passive_constructions / len(sentences) if sentences else 0.0

    citation_patterns = len(re.findall(r"\b\w+,\s*\d{4}\b|\bet\s+al\b", raw_text))
    features["citation_density"] = citation_patterns / len(sentences) if sentences else 0.0

    hedging_words = {"suggest", "appears", "likely", "unlikely", "possibly", "perhaps"}
    features["hedging_ratio"] = (sum(1 for w in raw_word_tokens if w in hedging_words) / len(raw_word_tokens) if raw_word_tokens else 0.0)

    return features

def build_handcrafted_matrix(raw_texts, clean_texts):
    feature_dicts = [extract_features(r, c) for r, c in zip(raw_texts, clean_texts)]
    feature_names = sorted(feature_dicts[0].keys())
    matrix = np.array([[fd[name] for name in feature_names] for fd in feature_dicts], dtype=float)
    return matrix, feature_names

In [ ]:
# Carregar artefatos exportados pelo modelo
with open('../models/subm2_model/vectorizer.pkl', 'rb') as f:
    vectorizer = pickle.load(f)
with open('../models/subm2_model/hc_stats.pkl', 'rb') as f:
    hc_stats = pickle.load(f)
    hc_mean, hc_std = hc_stats['mean'], hc_stats['std']
with open('../models/subm2_model/mapping.pkl', 'rb') as f:
    mapping = pickle.load(f)

# Ler dataset de teste
test_df = pd.read_csv('../data/dataset_teste.csv')
test_texts = test_df['Text'].fillna("").astype(str).tolist()

test_clean1 = [preprocess_text(t) for t in test_texts]
X_test_tfidf = vectorizer.transform(test_clean1).toarray()

test_clean2 = [preprocess_text_clean(t) for t in test_texts]
X_test_hc, _ = build_handcrafted_matrix(test_texts, test_clean2)
X_test_hc_std = (X_test_hc - hc_mean) / hc_std

X_test_full = np.hstack([X_test_tfidf, X_test_hc_std])

class TextDataset(Dataset):
    def __init__(self, X, y):
        if hasattr(X, "toarray"):
            X = X.toarray()
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self):
        return self.X.shape[0]
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

class LSTMClassifier(nn.Module):
    def __init__(self, input_dim, embed_dim=128, hidden_dim=128, n_classes=6):
        super().__init__()
        self.embedding = nn.Linear(input_dim, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, n_classes)
    def forward(self, x):
        x = self.embedding(x)
        x = x.unsqueeze(1)
        output, (h, c) = self.lstm(x)
        return self.fc(h[-1])

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = LSTMClassifier(input_dim=X_test_full.shape[1], n_classes=len(mapping)).to(device)
model.load_state_dict(torch.load('../models/subm2_model/lstm_model.pt', map_location=device))
model.eval()

# Gerar DataLoader
test_dataset = TextDataset(X_test_full, np.zeros(len(X_test_full)))
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

all_preds = []
with torch.no_grad():
    for X_batch, _ in test_loader:
        X_batch = X_batch.to(device)
        outputs = model(X_batch)
        preds = torch.argmax(outputs, dim=1)
        all_preds.extend(preds.cpu().numpy())

submission_df = test_df.copy()
submission_df['predicted_class'] = all_preds
reverse_mapping = {v: k for k, v in mapping.items()}
submission_df['Labels'] = submission_df['predicted_class'].map(reverse_mapping)

df_out = submission_df[['ID', 'Text', 'Labels']] if 'ID' in submission_df.columns else submission_df
file_out = 'subm2-MIA-B.csv'
df_out.to_csv(file_out, index=False)
print(f"Predições guardadas com sucesso em {file_out}")
df_out.head()